In [ ]:
!pip -q install timm scikit-learn pandas pillow tqdm openpyxl
from google.colab import drive, files
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize
import os, gc, shutil, zipfile, random, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

drive.mount('/content/drive')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['no_dr', 'mild', 'moderate', 'severe', 'proliferative_dr']
NUM_CLASSES = 5
IMG_SIZE = 384
RGB_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
RGB_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)
RGB_SCALE, MASK_SCALE, DROP = 0.75, 1.25, 0.45
EXPECTED_TEST_COUNT = 5269
RESULT_DIR = Path('/content/drive/MyDrive/DR_Project/zero_mask_all_four_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)


Mounted at /content/drive
Device: cuda


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT = DRIVE_ROOT / 'DR_Project'
STAGE1_W = PROJECT / 'stage1/weights/stage1_original_fixed_best_dr_b0.pth'
SPLIT_DIR = PROJECT / 'splits'
RGB_IMAGE_ROOT = DRIVE_ROOT / 'Dataset_extracted/colored_images/colored_images'
CHECKPOINTS = {
    'B0+B3+SE | predicted': PROJECT / 'stage3_stage2_masks_B1_B5_comparison/stage2_masks_B3_best.pth',
    'B0+B3 | predicted | no SE': PROJECT / 'stage3_ablation_B3_predicted_mask_without_SE/stage3_B3_best.pth',
    'B0+B4+SE | refined': PROJECT / 'stage3_b1_b5_comparison_with_val_loss/stage3_B4_best.pth',
    'B0+B4 | refined | no SE': PROJECT / 'stage3_ablation_B4_without_SE/stage3_B4_no_SE_best.pth',
}

PREDICTED_MASK_ZIP = ''  # ZIP containing a folder named stage2_masks
REFINED_MASK_ZIP = ''    # original stage3_local.zip with CSV + GAN-refined masks

required_paths = {'Stage 1 B0 weights': STAGE1_W, 'Stage 1 split folder': SPLIT_DIR,
                  'DR2015 RGB folder': RGB_IMAGE_ROOT, **CHECKPOINTS}
missing = {k: str(v) for k,v in required_paths.items() if not Path(v).exists()}
if missing:
    print('MISSING FILES (check Drive mount and adjust paths in this cell):')
    for k,v in missing.items(): print('  ',k,'=>',v)
    raise FileNotFoundError('One or more required existing checkpoints/data folders are missing. No new training is performed.')
for k, v in CHECKPOINTS.items(): print('FOUND',k,':',v)


FOUND B0+B3+SE | predicted : /content/drive/MyDrive/DR_Project/stage3_stage2_masks_B1_B5_comparison/stage2_masks_B3_best.pth
FOUND B0+B3 | predicted | no SE : /content/drive/MyDrive/DR_Project/stage3_ablation_B3_predicted_mask_without_SE/stage3_B3_best.pth
FOUND B0+B4+SE | refined : /content/drive/MyDrive/DR_Project/stage3_b1_b5_comparison_with_val_loss/stage3_B4_best.pth
FOUND B0+B4 | refined | no SE : /content/drive/MyDrive/DR_Project/stage3_ablation_B4_without_SE/stage3_B4_no_SE_best.pth


In [ ]:

WORK = Path('/content/zero_mask_four_models_data')
WORK.mkdir(parents=True, exist_ok=True)

def prepare_archive(existing_path, title, subfolder):
    target = WORK / subfolder
    if existing_path and Path(existing_path).is_dir():
        print(title, '- using folder:', existing_path)
        return Path(existing_path)
    if target.exists() and any(target.rglob('*.png')):
        print(title, '- reusing extracted folder:', target)
        return target
    if existing_path and Path(existing_path).is_file():
        zip_path = Path(existing_path)
    else:
        print('\nPlease upload',title, '(the SAME archive from the original notebook).')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Upload exactly one ZIP file for: '+title)
        zip_path = Path('/content') / next(iter(uploaded))
    if not zipfile.is_zipfile(zip_path):
        raise ValueError('Not a ZIP archive: '+str(zip_path))
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        # Avoid ZIP path traversal when extracting user-supplied files.
        base = target.resolve()
        for member in archive.infolist():
            dest = (base / member.filename).resolve()
            if dest != base and base not in dest.parents:
                raise ValueError('Unsafe ZIP entry: '+member.filename)
        archive.extractall(target)
    print(title, '- extracted to:',target)
    return target

PRED_ROOT = prepare_archive(PREDICTED_MASK_ZIP, 'predicted Stage 2 mask ZIP', 'predicted')
REF_ROOT = prepare_archive(REFINED_MASK_ZIP, 'GAN-refined Stage 3 ZIP', 'refined')



Please upload predicted Stage 2 mask ZIP (the SAME archive from the original notebook).


Saving stage2_masks.zip to stage2_masks.zip
predicted Stage 2 mask ZIP - extracted to: /content/zero_mask_four_models_data/predicted

Please upload GAN-refined Stage 3 ZIP (the SAME archive from the original notebook).


Saving stage3_masks.zip to stage3_masks.zip
GAN-refined Stage 3 ZIP - extracted to: /content/zero_mask_four_models_data/refined


In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
SUFFIXES = ['_stage2_mask', '_stage2', '_vessel_mask', '_vessel', '_mask',
            '_prob', '_pred', '_s2', '_stage3', '_gan', '_refined']

def canonical_stem(name):
    s = Path(str(name)).stem.lower()
    changed = True
    while changed:
        changed = False
        for suffix in SUFFIXES:
            if s.endswith(suffix): s = s[:-len(suffix)]; changed = True
    return s

def image_index(paths, label):
    result = {}
    for p in sorted(paths):
        if p.suffix.lower() not in IMAGE_EXTENSIONS: continue
        stem = canonical_stem(p.name)
        if stem in result and result[stem] != p:
            raise RuntimeError(f'Ambiguous {label} stem {stem}: {result[stem]} and {p}. Please provide a single mask folder.')
        result[stem] = p
    return result

def saved_names(name):
    p = SPLIT_DIR / name
    if not p.is_file(): raise FileNotFoundError(p)
    return [x.strip() for x in p.read_text().splitlines() if x.strip()]
train_names, val_names, test_names = (set(saved_names('stage1_train_names.txt')),
                                    set(saved_names('stage1_val_names.txt')),
                                    set(saved_names('stage1_test_names.txt')))
assert len(test_names) == EXPECTED_TEST_COUNT, f'Expected 5269 Stage 1 test images, found {len(test_names)}.'
assert not (train_names & test_names or val_names & test_names or val_names & train_names)
print('Stage 1 saved test IDs:',len(test_names))


pred_dirs = [d for d in PRED_ROOT.rglob('*') if d.is_dir() and d.name.lower() == 'stage2_masks']
if PRED_ROOT.name.lower() == 'stage2_masks': pred_dirs.insert(0, PRED_ROOT)
if len(pred_dirs) != 1:
    raise RuntimeError(f'Expected exactly one stage2_masks folder in predicted ZIP, found {len(pred_dirs)}: {pred_dirs[:5]}')
PRED_MASK_DIR = pred_dirs[0]
pred_index = image_index((p for p in PRED_MASK_DIR.rglob('*') if p.is_file()), 'predicted masks')
print('Stage 2 predicted masks found:',len(pred_index))


refined_candidates = []
for f in REF_ROOT.rglob('*.csv'):
    try:
        cols = set(pd.read_csv(f, nrows=0).columns)
        if {'img_path','label','stage3_mask_path'} <= cols:
            refined_candidates.append(f)
    except Exception: pass
if len(refined_candidates) != 1:
    raise RuntimeError(f'Expected one original GAN-refinement Stage 3 manifest CSV; found {len(refined_candidates)}: {refined_candidates[:8]}')
ref_manifest = pd.read_csv(refined_candidates[0])
ref_manifest['basename'] = ref_manifest['img_path'].map(lambda p: Path(str(p)).name)
ref_manifest = ref_manifest[ref_manifest['basename'].isin(test_names)].copy()
assert len(ref_manifest)==EXPECTED_TEST_COUNT, f'Refined manifest test rows: {len(ref_manifest)}, expected 5269.'
assert ref_manifest['basename'].nunique() == EXPECTED_TEST_COUNT, 'Repeated filenames in refined manifest.'
assert set(ref_manifest['basename']) == test_names, 'Refined test IDs do not match Stage 1 saved test IDs.'
ref_manifest['label'] = ref_manifest['label'].astype(int)
assert set(ref_manifest['label'].unique()) <= set(range(NUM_CLASSES)), 'Unexpected DR label.'


def resolve_refined_mask(original_path):
    original_path = str(original_path).strip()
    if Path(original_path).is_file(): return Path(original_path)
    normalized = original_path.replace('\\\\','/')
    for anchor in ('stage3_local/', 'stage3_local_extracted/'):
        if anchor in normalized:
            rel = normalized.split(anchor, 1)[1]
            found = [q for q in (REF_ROOT / 'stage3_local' / rel,
                                 REF_ROOT / rel, REF_ROOT / 'stage3_local_extracted' / rel)
                     if q.is_file()]
            if len(found)==1: return found[0]
    candidate = REF_ROOT / normalized.lstrip('/')
    if candidate.is_file(): return candidate

    matches = list(REF_ROOT.rglob(Path(original_path).name))
    matches = [m for m in matches if m.is_file() and m.suffix.lower() in IMAGE_EXTENSIONS]
    if len(matches)==1: return matches[0]
    raise FileNotFoundError(f'Cannot uniquely resolve original GAN-refined mask: {original_path}; candidates={matches[:4]}')

# Match RGB images by the exact original basename.
test_lookup = set(test_names)
rgb_index = {}
for p in RGB_IMAGE_ROOT.rglob('*'):
    if p.is_file() and p.name in test_lookup:
        if p.name in rgb_index: raise RuntimeError('Duplicate RGB basename: '+p.name)
        rgb_index[p.name] = p
assert set(rgb_index) == test_names, f'Missing RGB test files: {sorted(test_names-set(rgb_index))[:10]}'

manifest = ref_manifest[['basename','label','stage3_mask_path']].copy()
manifest = manifest.sort_values('basename').reset_index(drop=True)
manifest['image_path'] = manifest['basename'].map(lambda nm: str(rgb_index[nm]))
manifest['predicted_mask_path'] = manifest['basename'].map(lambda nm: str(pred_index.get(canonical_stem(nm),'') ))
missing_pred = manifest[~manifest['predicted_mask_path'].map(os.path.isfile)]
if len(missing_pred):
    print(missing_pred[['basename','predicted_mask_path']].head(10).to_string(index=False))
    raise RuntimeError(f'{len(missing_pred)} predicted masks missing for original test images.')
manifest['refined_mask_path'] = manifest['stage3_mask_path'].map(lambda v: str(resolve_refined_mask(v)))
assert manifest['refined_mask_path'].map(os.path.isfile).all()
assert len(manifest)==EXPECTED_TEST_COUNT
assert set(manifest['basename'])==test_names
manifest.to_csv(RESULT_DIR / 'four_model_same_test_manifest.csv', index=False)
print('\nVerified exactly',len(manifest),'identical test image IDs for all four models.')
print('Class support:\n',manifest['label'].map(dict(enumerate(CLASS_NAMES))).value_counts().to_string())
print('Saved test manifest to',RESULT_DIR)


Stage 1 saved test IDs: 5269
Stage 2 predicted masks found: 35126

Verified exactly 5269 identical test image IDs for all four models.
Class support:
 label
no_dr               3872
moderate             794
mild                 366
severe               131
proliferative_dr     106
Saved test manifest to /content/drive/MyDrive/DR_Project/zero_mask_all_four_results


In [ ]:

class SEBlock(nn.Module):

    def __init__(
        self,
        in_dim,
        reduction=16
    ):

        super().__init__()

        hidden_dim = max(
            1,
            in_dim // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(
                in_dim,
                hidden_dim,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                hidden_dim,
                in_dim,
                bias=False
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        x
    ):

        weights = self.fc(
            x
        )

        return (
            x
            *
            weights
        )

class _MaskEncoderProxy:

    def __init__(
        self,
        model
    ):

        self.enc3 = (
            model.blocks[-2]
        )

        self.enc4 = (
            model.blocks[-1]
        )

class Stage3EfficientNetB0BxSE(nn.Module):

    def __init__(
        self,
        num_classes,
        stage1_weight_path,
        mask_model_name,
        mask_pretrained=True
    ):

        super().__init__()


        self.rgb_scale = (
            RGB_SCALE
        )

        self.mask_scale = (
            MASK_SCALE
        )

        self.mask_model_name = (
            mask_model_name
        )


        rgb_model = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )


        stage1_state = torch.load(
            stage1_weight_path,
            map_location="cpu"
        )


        if (
            isinstance(
                stage1_state,
                dict
            )
            and
            "state_dict"
            in stage1_state
        ):

            stage1_state = (
                stage1_state[
                    "state_dict"
                ]
            )


        if (
            isinstance(
                stage1_state,
                dict
            )
            and
            "model_state_dict"
            in stage1_state
        ):

            stage1_state = (
                stage1_state[
                    "model_state_dict"
                ]
            )


        stage1_state = {

            key:
                value

            for (
                key,
                value
            )
            in stage1_state.items()

            if not key.startswith(
                "classifier"
            )
        }


        rgb_model.load_state_dict(
            stage1_state,
            strict=False
        )


        for parameter in (
            rgb_model.parameters()
        ):

            parameter.requires_grad = False


        self.rgb_branch = (
            rgb_model
        )


        mask_model = timm.create_model(
            mask_model_name,
            pretrained=mask_pretrained,
            num_classes=0,
            global_pool="avg"
        )


        old_conv = (
            mask_model.conv_stem
        )


        new_conv = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=
                old_conv.kernel_size,
            stride=
                old_conv.stride,
            padding=
                old_conv.padding,
            dilation=
                old_conv.dilation,
            groups=
                old_conv.groups,
            bias=False
        )


        with torch.no_grad():

            new_conv.weight.copy_(
                old_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )


        mask_model.conv_stem = (
            new_conv
        )


        if hasattr(
            mask_model,
            "set_grad_checkpointing"
        ):

            try:

                mask_model.set_grad_checkpointing(
                    True
                )

            except Exception:

                pass


        self.mask_branch = (
            mask_model
        )


        self._mask_encoder_proxy = (
            _MaskEncoderProxy(
                mask_model
            )
        )


        self.RGB_DIM = int(
            rgb_model.num_features
        )


        self.MASK_DIM = int(
            mask_model.num_features
        )


        self.FUSE_DIM = (
            self.RGB_DIM
            +
            self.MASK_DIM
        )


        self.se = SEBlock(
            self.FUSE_DIM,
            reduction=16
        )


        self.head = nn.Sequential(

            nn.Dropout(
                DROP
            ),

            nn.Linear(
                self.FUSE_DIM,
                num_classes
            )
        )


    @property
    def mask_encoder(
        self
    ):

        return (
            self._mask_encoder_proxy
        )


    def _features(
        self,
        rgb,
        mask
    ):

        with torch.no_grad():

            rgb_features = (
                self.rgb_branch(
                    rgb
                )
            )

        mask_features = (
            self.mask_branch(
                mask
            )
        )


        rgb_features = (
            self.rgb_scale
            *
            rgb_features
        )


        mask_features = (
            self.mask_scale
            *
            mask_features
        )


        fused_features = torch.cat(
            [
                rgb_features,
                mask_features
            ],
            dim=1
        )


        attended_features = (
            self.se(
                fused_features
            )
        )


        return (
            attended_features,
            rgb_features,
            mask_features
        )


    def forward(
        self,
        rgb,
        mask
    ):

        (
            attended_features,
            _,
            _
        ) = self._features(
            rgb,
            mask
        )


        logits = self.head(
            attended_features
        )


        return logits


    def forward_with_parts(
        self,
        rgb,
        mask
    ):

        (
            attended_features,
            rgb_features,
            mask_features
        ) = self._features(
            rgb,
            mask
        )


        logits = self.head(
            attended_features
        )


        return (
            logits,
            mask_features
        )


    def forward_rgb_only(
        self,
        rgb,
        mask=None
    ):

        with torch.no_grad():

            rgb_features = (
                self.rgb_scale
                *
                self.rgb_branch(
                    rgb
                )
            )


        zero_mask_features = (
            torch.zeros(
                rgb_features.size(0),
                self.MASK_DIM,
                device=
                    rgb_features.device,
                dtype=
                    rgb_features.dtype
            )
        )


        fused = torch.cat(
            [
                rgb_features,
                zero_mask_features
            ],
            dim=1
        )


        attended = self.se(
            fused
        )


        return self.head(
            attended
        )


    def forward_mask_only(
        self,
        rgb,
        mask
    ):

        mask_features = (
            self.mask_scale
            *
            self.mask_branch(
                mask
            )
        )


        zero_rgb_features = (
            torch.zeros(
                mask_features.size(0),
                self.RGB_DIM,
                device=
                    mask_features.device,
                dtype=
                    mask_features.dtype
            )
        )


        fused = torch.cat(
            [
                zero_rgb_features,
                mask_features
            ],
            dim=1
        )


        attended = self.se(
            fused
        )


        return self.head(
            attended
        )

class Stage3EfficientNetB0BxWithoutSE(nn.Module):

    def __init__(
        self,
        num_classes,
        stage1_weight_path,
        mask_model_name,
        mask_pretrained=True
    ):

        super().__init__()


        self.rgb_scale = (
            RGB_SCALE
        )

        self.mask_scale = (
            MASK_SCALE
        )

        self.mask_model_name = (
            mask_model_name
        )


        rgb_model = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )


        stage1_state = torch.load(
            stage1_weight_path,
            map_location="cpu"
        )


        if (
            isinstance(
                stage1_state,
                dict
            )
            and
            "state_dict"
            in stage1_state
        ):

            stage1_state = (
                stage1_state[
                    "state_dict"
                ]
            )


        if (
            isinstance(
                stage1_state,
                dict
            )
            and
            "model_state_dict"
            in stage1_state
        ):

            stage1_state = (
                stage1_state[
                    "model_state_dict"
                ]
            )


        stage1_state = {

            key:
                value

            for (
                key,
                value
            )
            in stage1_state.items()

            if not key.startswith(
                "classifier"
            )
        }


        rgb_model.load_state_dict(
            stage1_state,
            strict=False
        )


        for parameter in (
            rgb_model.parameters()
        ):

            parameter.requires_grad = False


        self.rgb_branch = (
            rgb_model
        )


        mask_model = timm.create_model(
            mask_model_name,
            pretrained=mask_pretrained,
            num_classes=0,
            global_pool="avg"
        )


        old_conv = (
            mask_model.conv_stem
        )


        new_conv = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=
                old_conv.kernel_size,
            stride=
                old_conv.stride,
            padding=
                old_conv.padding,
            dilation=
                old_conv.dilation,
            groups=
                old_conv.groups,
            bias=False
        )


        with torch.no_grad():

            new_conv.weight.copy_(
                old_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )


        mask_model.conv_stem = (
            new_conv
        )


        if hasattr(
            mask_model,
            "set_grad_checkpointing"
        ):

            try:

                mask_model.set_grad_checkpointing(
                    True
                )

            except Exception:

                pass


        self.mask_branch = (
            mask_model
        )


        self._mask_encoder_proxy = (
            _MaskEncoderProxy(
                mask_model
            )
        )


        self.RGB_DIM = int(
            rgb_model.num_features
        )


        self.MASK_DIM = int(
            mask_model.num_features
        )


        self.FUSE_DIM = (
            self.RGB_DIM
            +
            self.MASK_DIM
        )



        self.se = nn.Identity()


        self.head = nn.Sequential(

            nn.Dropout(
                DROP
            ),

            nn.Linear(
                self.FUSE_DIM,
                num_classes
            )
        )


    @property
    def mask_encoder(
        self
    ):

        return (
            self._mask_encoder_proxy
        )


    def _features(
        self,
        rgb,
        mask
    ):

        with torch.no_grad():

            rgb_features = (
                self.rgb_branch(
                    rgb
                )
            )

        mask_features = (
            self.mask_branch(
                mask
            )
        )


        rgb_features = (
            self.rgb_scale
            *
            rgb_features
        )


        mask_features = (
            self.mask_scale
            *
            mask_features
        )


        fused_features = torch.cat(
            [
                rgb_features,
                mask_features
            ],
            dim=1
        )


        fused_features = (
            self.se(
                fused_features
            )
        )


        return (
            fused_features,
            rgb_features,
            mask_features
        )


    def forward(
        self,
        rgb,
        mask
    ):

        (
            fused_features,
            _,
            _
        ) = self._features(
            rgb,
            mask
        )


        logits = self.head(
            fused_features
        )


        return logits


    def forward_with_parts(
        self,
        rgb,
        mask
    ):

        (
            fused_features,
            rgb_features,
            mask_features
        ) = self._features(
            rgb,
            mask
        )


        logits = self.head(
            fused_features
        )


        return (
            logits,
            mask_features
        )


    def forward_rgb_only(
        self,
        rgb,
        mask=None
    ):

        with torch.no_grad():

            rgb_features = (
                self.rgb_scale
                *
                self.rgb_branch(
                    rgb
                )
            )


        zero_mask_features = (
            torch.zeros(
                rgb_features.size(0),
                self.MASK_DIM,
                device=
                    rgb_features.device,
                dtype=
                    rgb_features.dtype
            )
        )


        fused = torch.cat(
            [
                rgb_features,
                zero_mask_features
            ],
            dim=1
        )


        fused = self.se(
            fused
        )


        return self.head(
            fused
        )


    def forward_mask_only(
        self,
        rgb,
        mask
    ):

        mask_features = (
            self.mask_scale
            *
            self.mask_branch(
                mask
            )
        )


        zero_rgb_features = (
            torch.zeros(
                mask_features.size(0),
                self.RGB_DIM,
                device=
                    mask_features.device,
                dtype=
                    mask_features.dtype
            )
        )


        fused = torch.cat(
            [
                zero_rgb_features,
                mask_features
            ],
            dim=1
        )


        fused = self.se(
            fused
        )


        return self.head(
            fused
        )

def load_stage1_feature_weights(path):
    ckpt = torch.load(path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise RuntimeError("Unsupported Stage1 checkpoint format.")

    clean = {}

    for k, v in sd.items():
        k = k.replace("module.", "")
        k = k.replace("_orig_mod.", "")

        if k.startswith("classifier"):
            continue

        clean[k] = v

    return clean

class Stage3Stage2MaskB0BxSE(nn.Module):
    def __init__(
        self,
        mask_model_name,
        num_classes,
        stage1_weight_path,
        drop=0.45,
        rgb_scale=0.75,
        mask_scale=1.25,
        mask_pretrained=True
    ):
        super().__init__()

        self.mask_model_name = mask_model_name
        self.rgb_scale = rgb_scale
        self.mask_scale = mask_scale

        rgb_net = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )

        rgb_sd = load_stage1_feature_weights(stage1_weight_path)
        missing, unexpected = rgb_net.load_state_dict(rgb_sd, strict=False)

        meaningful_missing = [
            k for k in missing
            if not k.startswith("classifier")
        ]

        meaningful_unexpected = [
            k for k in unexpected
            if not k.startswith("classifier")
        ]

        if meaningful_missing or meaningful_unexpected:
            print("RGB missing:", meaningful_missing[:20])
            print("RGB unexpected:", meaningful_unexpected[:20])
            raise RuntimeError("Stage1 feature checkpoint did not match B0 RGB branch.")

        for p in rgb_net.parameters():
            p.requires_grad = False

        self.rgb_branch = rgb_net

        mask_net = timm.create_model(
            mask_model_name,
            pretrained=mask_pretrained,
            num_classes=0,
            global_pool="avg"
        )

        old_conv = mask_net.conv_stem

        new_conv = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            dilation=old_conv.dilation,
            groups=old_conv.groups,
            bias=False
        )

        with torch.no_grad():
            new_conv.weight.copy_(
                old_conv.weight.mean(dim=1, keepdim=True)
            )

        mask_net.conv_stem = new_conv

        self.mask_branch = mask_net
        self._mask_encoder_proxy = _MaskEncoderProxy(mask_net)

        self.RGB_DIM = int(rgb_net.num_features)
        self.MASK_DIM = int(mask_net.num_features)
        self.FUSE_DIM = self.RGB_DIM + self.MASK_DIM

        self.se = SEBlock(
            self.FUSE_DIM,
            reduction=16
        )

        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(self.FUSE_DIM, num_classes)
        )

    @property
    def mask_encoder(self):
        return self._mask_encoder_proxy

    def _features(self, rgb, mask):
        with torch.no_grad():
            rgb_feat = self.rgb_branch(rgb)

        mask_feat = self.mask_branch(mask)

        rgb_feat = self.rgb_scale * rgb_feat
        mask_feat = self.mask_scale * mask_feat

        attended = self.se(
            torch.cat(
                [rgb_feat, mask_feat],
                dim=1
            )
        )

        return attended, rgb_feat, mask_feat

    def forward(self, rgb, mask):
        attended, _, _ = self._features(rgb, mask)
        return self.head(attended)

    def forward_with_parts(self, rgb, mask):
        attended, rgb_feat, mask_feat = self._features(rgb, mask)

        logits = self.head(attended)

        zero_mask_feat = torch.zeros_like(mask_feat)

        rgb_only_attended = self.se(
            torch.cat(
                [rgb_feat, zero_mask_feat],
                dim=1
            )
        )

        rgb_only_logits = self.head(rgb_only_attended)

        return (
            logits,
            attended,
            mask_feat,
            rgb_only_logits
        )

    def forward_rgb_only(self, rgb, mask):
        with torch.no_grad():
            rgb_feat = self.rgb_branch(rgb)

        rgb_feat = self.rgb_scale * rgb_feat

        zero_mask = torch.zeros(
            rgb_feat.shape[0],
            self.MASK_DIM,
            device=rgb_feat.device,
            dtype=rgb_feat.dtype
        )

        attended = self.se(
            torch.cat(
                [rgb_feat, zero_mask],
                dim=1
            )
        )

        return self.head(attended)

    def forward_mask_only(self, rgb, mask):
        mask_feat = self.mask_branch(mask)
        mask_feat = self.mask_scale * mask_feat

        zero_rgb = torch.zeros(
            mask_feat.shape[0],
            self.RGB_DIM,
            device=mask_feat.device,
            dtype=mask_feat.dtype
        )

        attended = self.se(
            torch.cat(
                [zero_rgb, mask_feat],
                dim=1
            )
        )

        return self.head(attended)

class Stage3EfficientNetB0BxNoSE(Stage3EfficientNetB0BxSE):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.se = nn.Identity()


In [ ]:

class FixedStage3TestDataset(Dataset):
    def __init__(self, frame, mask_type):
        self.df = frame.reset_index(drop=True)
        self.mask_column = 'predicted_mask_path' if mask_type=='predicted' else 'refined_mask_path'
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row['image_path']) as im:
            im = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
            rgb = np.asarray(im, dtype=np.float32) / 255.0
        rgb = np.transpose((rgb-RGB_MEAN)/RGB_STD, (2,0,1)).copy()
        with Image.open(row[self.mask_column]) as im:
            im = im.convert('L').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST)
            mask = (np.asarray(im, dtype=np.float32)/255.0 > 0.5).astype(np.float32)
        return (torch.from_numpy(rgb),torch.from_numpy(mask[None].copy()),
                int(row['label']),str(row['basename']))

DATASETS = {
    'predicted': FixedStage3TestDataset(manifest,'predicted'),
    'refined': FixedStage3TestDataset(manifest,'refined'),
}
CONFIGS = [
    ('B0+B3+SE | predicted', 'B3', True, 'predicted'),
    ('B0+B3 | predicted | no SE', 'B3', False, 'predicted'),
    ('B0+B4+SE | refined', 'B4', True, 'refined'),
    ('B0+B4 | refined | no SE', 'B4', False, 'refined'),
]
BACKBONES = {'B3':'tf_efficientnet_b3.ns_jft_in1k','B4':'tf_efficientnet_b4.ns_jft_in1k'}
BATCH_SIZES = {'B3':16,'B4':8}  # Original per-model micro-batches for these backbones

def load_exact_stage3_model(model_name, backbone, with_se):
    checkpoint = CHECKPOINTS[model_name]
    if backbone == 'B3' and with_se:
        model = Stage3Stage2MaskB0BxSE(mask_model_name=BACKBONES[backbone],
            num_classes=NUM_CLASSES,stage1_weight_path=str(STAGE1_W),
            mask_pretrained=False,drop=DROP,rgb_scale=RGB_SCALE,mask_scale=MASK_SCALE)
    elif backbone == 'B3':
        model = Stage3EfficientNetB0BxWithoutSE(num_classes=NUM_CLASSES,
            stage1_weight_path=str(STAGE1_W),mask_model_name=BACKBONES[backbone],mask_pretrained=False)
    elif with_se:
        model = Stage3EfficientNetB0BxSE(num_classes=NUM_CLASSES,
            stage1_weight_path=str(STAGE1_W),mask_model_name=BACKBONES[backbone],mask_pretrained=False)
    else:
        model = Stage3EfficientNetB0BxNoSE(num_classes=NUM_CLASSES,
            stage1_weight_path=str(STAGE1_W),mask_model_name=BACKBONES[backbone],mask_pretrained=False)
    state = torch.load(checkpoint,map_location='cpu',weights_only=True)
    if isinstance(state,dict) and 'model_state_dict' in state: state=state['model_state_dict']
    elif isinstance(state,dict) and 'state_dict' in state: state=state['state_dict']
    model.load_state_dict(state,strict=True)
    return model.to(DEVICE).eval()

@torch.inference_mode()
def evaluate_actual_and_zero(model, loader):
    ids, y, probs_actual, probs_zero = [],[],[],[]
    model.eval()
    for rgb, mask, labels, filenames in tqdm(loader,desc='Test actual and zero masks'):
        rgb = rgb.to(DEVICE,non_blocking=True)
        mask = mask.to(DEVICE,non_blocking=True)
        # Pass zero IMAGE mask through the vessel branch, not zeroed feature embeddings.
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=='cuda')):
            logits_actual = model(rgb,mask)
            logits_zero = model(rgb,torch.zeros_like(mask))
        probs_actual.append(torch.softmax(logits_actual.float(),dim=1).cpu().numpy())
        probs_zero.append(torch.softmax(logits_zero.float(),dim=1).cpu().numpy())
        y.extend(labels.numpy().tolist()); ids.extend(list(filenames))
    return (np.asarray(ids),np.asarray(y,dtype=int),
            np.concatenate(probs_actual),np.concatenate(probs_zero))


In [ ]:

PREDICTIONS = {}
for model_name,backbone,with_se,mask_type in CONFIGS:
    print('\n'+'='*95+'\n',model_name,'\n'+'='*95)
    model = load_exact_stage3_model(model_name, backbone, with_se)
    loader = DataLoader(DATASETS[mask_type],batch_size=BATCH_SIZES[backbone],
                        shuffle=False,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
    ids, labels, prob_actual, prob_zero = evaluate_actual_and_zero(model,loader)
    if len(labels)!=EXPECTED_TEST_COUNT: raise RuntimeError('Incomplete evaluation for '+model_name)
    if list(ids) != manifest['basename'].tolist(): raise RuntimeError('Test ordering differs!')
    if not np.array_equal(labels,manifest['label'].to_numpy(dtype=int)):
        raise RuntimeError('Test labels differ!')
    PREDICTIONS[model_name] = {'ids':ids,'labels':labels,
                               'actual':prob_actual,'zero':prob_zero}
    print('Completed: ',len(labels),'test images (real mask AND zero mask).')
    del model,loader; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
print('\nALL FOUR MODELS COMPLETED.')



 B0+B3+SE | predicted 


Test actual and zero masks:   0%|          | 0/330 [00:00<?, ?it/s]

Completed:  5269 test images (real mask AND zero mask).

 B0+B3 | predicted | no SE 


Test actual and zero masks:   0%|          | 0/330 [00:00<?, ?it/s]

Completed:  5269 test images (real mask AND zero mask).

 B0+B4+SE | refined 


Test actual and zero masks:   0%|          | 0/659 [00:00<?, ?it/s]

Completed:  5269 test images (real mask AND zero mask).

 B0+B4 | refined | no SE 


Test actual and zero masks:   0%|          | 0/659 [00:00<?, ?it/s]

Completed:  5269 test images (real mask AND zero mask).

ALL FOUR MODELS COMPLETED.


In [ ]:
# Calculate test metrics and per-severity actual-class mean probabilities.
OVERALL, PER_CLASS_SCORE, PER_CLASS_METRIC, CHANGES, PRED_ROWS, CONFUSIONS = [],[],[],[],[],[]
LABELS=np.arange(NUM_CLASSES)

def safe_macro_auc(y,prob):
    try:
        return float(roc_auc_score(label_binarize(y,classes=LABELS),prob,average='macro',multi_class='ovr'))
    except ValueError: return np.nan

for model_name, backbone, with_se, mask_type in CONFIGS:
    result = PREDICTIONS[model_name]
    y = result['labels']; ids=result['ids']
    probs_a,probs_z=result['actual'],result['zero']
    pa,pz=probs_a.argmax(axis=1),probs_z.argmax(axis=1)
    true_score_a=probs_a[np.arange(len(y)),y]
    true_score_z=probs_z[np.arange(len(y)),y]
    for condition,pred,prob in [('actual_mask',pa,probs_a),('zero_mask',pz,probs_z)]:
        OVERALL.append({'model':model_name,'vessel_backbone':backbone,'se':with_se,
             'mask_type':mask_type,'condition':condition,'support':len(y),
             'accuracy':accuracy_score(y,pred),
             'macro_f1':f1_score(y,pred,labels=LABELS,average='macro',zero_division=0),
             'weighted_f1':f1_score(y,pred,labels=LABELS,average='weighted',zero_division=0),
             'macro_precision':precision_score(y,pred,labels=LABELS,average='macro',zero_division=0),
             'macro_recall':recall_score(y,pred,labels=LABELS,average='macro',zero_division=0),
             'macro_auroc':safe_macro_auc(y,prob)})
        report=classification_report(y,pred,labels=LABELS,target_names=CLASS_NAMES,
                                     output_dict=True,zero_division=0)
        for cls in CLASS_NAMES:
            PER_CLASS_METRIC.append({'model':model_name,'condition':condition,
                                     'severity':cls,**report[cls]})
        cm=confusion_matrix(y,pred,labels=LABELS)
        for i in LABELS:
            for j in LABELS:
                CONFUSIONS.append({'model':model_name,'condition':condition,
                    'true_class':CLASS_NAMES[i],'pred_class':CLASS_NAMES[j],'count':int(cm[i,j])})
    for k,cls in enumerate(CLASS_NAMES):
        take=(y==k)
        score_a=float(true_score_a[take].mean())
        score_z=float(true_score_z[take].mean())
        PER_CLASS_SCORE.append({'model':model_name,'mask_type':mask_type,'se':with_se,
              'severity':cls,'support':int(take.sum()),'avg_score_with_mask':score_a,
              'avg_score_zero_mask':score_z,'difference_with_minus_zero':score_a-score_z,
              'pct_individual_scores_higher_with_mask':float(np.mean(true_score_a[take]>true_score_z[take])*100)})
    CHANGES.append({'model':model_name,'changed_predicted_class_count':int(np.sum(pa!=pz)),
                    'zero_mask_wrong_to_correct_count':int(np.sum((pa!=y)&(pz==y))),
                    'zero_mask_correct_to_wrong_count':int(np.sum((pa==y)&(pz!=y)))})
    for i,nm in enumerate(ids):
        row={'model':model_name,'image_id':nm,'true_class':CLASS_NAMES[y[i]],
             'predicted_with_mask':CLASS_NAMES[pa[i]],'predicted_zero_mask':CLASS_NAMES[pz[i]],
             'actual_class_score_with_mask':float(true_score_a[i]),
             'actual_class_score_zero_mask':float(true_score_z[i]),
             'score_difference':float(true_score_a[i]-true_score_z[i])}
        for k,cl in enumerate(CLASS_NAMES):
            row['with_mask_prob_'+cl]=float(probs_a[i,k])
            row['zero_mask_prob_'+cl]=float(probs_z[i,k])
        PRED_ROWS.append(row)

overall=pd.DataFrame(OVERALL)
per_score=pd.DataFrame(PER_CLASS_SCORE)
per_metrics=pd.DataFrame(PER_CLASS_METRIC)
changes=pd.DataFrame(CHANGES)
pred_rows=pd.DataFrame(PRED_ROWS)
confusions=pd.DataFrame(CONFUSIONS)

# Signed metric difference: value WITH vessel mask minus value WITH zero mask.
actual_metrics=overall[overall.condition=='actual_mask'].set_index('model')
zero_metrics=overall[overall.condition=='zero_mask'].set_index('model')
metric_names=['accuracy','macro_f1','weighted_f1','macro_precision','macro_recall','macro_auroc']
comparison=actual_metrics[['vessel_backbone','se','mask_type','support']].copy()
for metric in metric_names:
    comparison[metric+'_with_mask']=actual_metrics[metric]
    comparison[metric+'_zero_mask']=zero_metrics[metric]
    comparison[metric+'_difference']=actual_metrics[metric]-zero_metrics[metric]
comparison=comparison.reset_index()

files_to_save = {
    '01_overall_metrics_actual_and_zero.csv':overall,
    '02_metric_differences_with_minus_zero.csv':comparison,
    '03_average_actual_class_scores_by_severity.csv':per_score,
    '04_per_class_precision_recall_f1.csv':per_metrics,
    '05_prediction_switch_counts.csv':changes,
    '06_per_image_predictions_and_probabilities.csv':pred_rows,
    '07_confusion_matrix_long.csv':confusions,
}
for filename,dataframe in files_to_save.items():
    filepath=RESULT_DIR/filename
    dataframe.to_csv(filepath,index=False,float_format='%.10f')
    print('Saved:',filepath)


for mdl, exp in EXPECTED_ACCURACY.items():
    observed = float(actual_metrics.loc[mdl, 'accuracy'])
    if abs(observed - exp) > 0.002:
        warnings.warn(f'{mdl}: actual-mask accuracy {observed:.4f} does not match original report {exp:.4f}. Check inputs/checkpoint before using results.')

print('\n'+'='*110)
print('TABLE A: OVERALL CLASSIFICATION METRICS (WITH MASK vs ZERO MASK)')
print('='*110)
print(overall.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\n'+'='*110)
print('TABLE B: DIFFERENCE IN ACCURACY, MACRO-F1 AND MACRO RECALL (WITH - ZERO)')
print('='*110)
cols=['model','accuracy_with_mask','accuracy_zero_mask','accuracy_difference',
      'macro_f1_with_mask','macro_f1_zero_mask','macro_f1_difference',
      'macro_recall_with_mask','macro_recall_zero_mask','macro_recall_difference']
print(comparison[cols].to_string(index=False,float_format=lambda x:f'{x:.4f}'))

from IPython.display import display

def show_grouped_severity_scores(title, b4_name, b3_name, b4_label, b3_label, csv_name):
    column_labels = (
        'Stage 3 With Vessel Mask Input',
        'Stage 3 With Zero Vessel Mask Input',
        'Difference',
    )
    grouped = pd.DataFrame(index=pd.Index(
        ['No_dr', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
        name='DR Severity',
    ))
    for model_name, header in [(b4_name, b4_label), (b3_name, b3_label)]:
        extracted = (per_score.loc[per_score['model'].eq(model_name)]
                     .set_index('severity').reindex(CLASS_NAMES))
        if extracted['avg_score_with_mask'].isna().any():
            raise ValueError(f'Missing severity scores for {model_name}')
        for label, field in zip(column_labels, (
            'avg_score_with_mask', 'avg_score_zero_mask', 'difference_with_minus_zero'
        )):
            grouped[(header, label)] = extracted[field].to_numpy()
    grouped.columns = pd.MultiIndex.from_tuples(grouped.columns)

    # Keep numerical precision in the saved CSV; show exactly four decimal places.
    grouped.to_csv(RESULT_DIR / csv_name, float_format='%.10f')
    print('\n' + '=' * 110)
    print(title)
    print('=' * 110)
    display(grouped.style.format({
        (header, label): ('{:+.4f}' if label == 'Difference' else '{:.4f}')
        for header in (b4_label, b3_label) for label in column_labels
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ]))
    return grouped

scores_with_se = show_grouped_severity_scores(
    'TABLE C1: AVERAGE ACTUAL-CLASS SCORES BY DR SEVERITY — WITH SE ATTENTION',
    'B0+B4+SE | refined',
    'B0+B3+SE | predicted',
    'Stage 3 With Refinement (B0+B4+SE)',
    'Stage 3 Without Refinement (B0+B3+SE)',
    '08_grouped_severity_scores_with_SE.csv',
)
scores_without_se = show_grouped_severity_scores(
    'TABLE C2: AVERAGE ACTUAL-CLASS SCORES BY DR SEVERITY — WITHOUT SE ATTENTION',
    'B0+B4 | refined | no SE',
    'B0+B3 | predicted | no SE',
    'Stage 3 With Refinement (B0+B4, without SE)',
    'Stage 3 Without Refinement (B0+B3, without SE)',
    '09_grouped_severity_scores_without_SE.csv',
)
print('\n'+'='*110)
print('TABLE D: PER-CLASS PRECISION / RECALL / F1')
print('='*110)
print(per_metrics[['model','condition','severity','precision','recall','f1-score','support']].to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\n'+'='*110)
print('PREDICTION SWITCH COUNTS')
print('='*110)
print(changes.to_string(index=False))
print('\nAll four models and both test conditions finished. Results saved in:',RESULT_DIR)


Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/01_overall_metrics_actual_and_zero.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/02_metric_differences_with_minus_zero.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/03_average_actual_class_scores_by_severity.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/04_per_class_precision_recall_f1.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/05_prediction_switch_counts.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/06_per_image_predictions_and_probabilities.csv
Saved: /content/drive/MyDrive/DR_Project/zero_mask_all_four_results/07_confusion_matrix_long.csv

TABLE A: OVERALL CLASSIFICATION METRICS (WITH MASK vs ZERO MASK)
                    model vessel_backbone    se mask_type   condition  support  accuracy  macro_f1  weighted_f1  macro_precision  macro_recall  macro_auroc
     B0+B3+SE | predicted    


TABLE C2: AVERAGE ACTUAL-CLASS SCORES BY DR SEVERITY — WITHOUT SE ATTENTION



TABLE D: PER-CLASS PRECISION / RECALL / F1
                    model   condition         severity  precision  recall  f1-score   support
     B0+B3+SE | predicted actual_mask            no_dr     0.8456  0.8587    0.8521 3872.0000
     B0+B3+SE | predicted actual_mask             mild     0.1239  0.1475    0.1347  366.0000
     B0+B3+SE | predicted actual_mask         moderate     0.5217  0.4698    0.4944  794.0000
     B0+B3+SE | predicted actual_mask           severe     0.4623  0.3740    0.4135  131.0000
     B0+B3+SE | predicted actual_mask proliferative_dr     0.7250  0.5472    0.6237  106.0000
     B0+B3+SE | predicted   zero_mask            no_dr     0.8443  0.8784    0.8610 3872.0000
     B0+B3+SE | predicted   zero_mask             mild     0.1247  0.1448    0.1340  366.0000
     B0+B3+SE | predicted   zero_mask         moderate     0.5590  0.4118    0.4743  794.0000
     B0+B3+SE | predicted   zero_mask           severe     0.4161  0.4733    0.4429  131.0000
     B0+B3+SE | 

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.dimensions import ColumnDimension
from openpyxl.formatting.rule import CellIsRule
from openpyxl.workbook.properties import CalcProperties
from datetime import datetime

EXCEL_FILE = RESULT_DIR / 'Zero_Mask_Four_Models_Full_Results.xlsx'
EXCEL_FILE.parent.mkdir(parents=True, exist_ok=True)

wb = Workbook()
wb.remove(wb.active)

DARK_BLUE = '18324B'
MID_BLUE = 'DCE9F1'
LIGHT_BLUE = 'ECF5F8'
WHITE = 'FFFFFF'
GREEN = '17724D'
GRAY = '677483'
ORANGE = 'AB6516'
RED = '9D3636'
NUM4 = '0.0000'
SIGNED4 = '+0.0000;-0.0000;0.0000'
INT_FMT = '0'


def safe_excel(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if pd.isna(v):
        return None
    return v


def write_table(sheet_name, frame, title, cols=None, signed_cols=()):
    ws = wb.create_sheet(sheet_name)
    if cols is not None:
        frame = frame.loc[:, cols]
    ncol = len(frame.columns)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=ncol)
    top = ws.cell(1, 1, title)
    top.fill = PatternFill('solid', fgColor=DARK_BLUE)
    top.font = Font(name='Aptos', size=13, bold=True, color=WHITE)
    top.alignment = Alignment(vertical='center')
    ws.row_dimensions[1].height = 27
    for j, head in enumerate(frame.columns, 1):
        cell = ws.cell(3, j, str(head))
        cell.fill = PatternFill('solid', fgColor=MID_BLUE)
        cell.font = Font(name='Aptos', bold=True, color=DARK_BLUE, size=10)
        cell.alignment = Alignment(vertical='center', wrap_text=True)
        h = str(head).lower()
        ws.column_dimensions[get_column_letter(j)].width = min(40, max(15, len(str(head)) + 3))
    ws.row_dimensions[3].height = 32
    for i, row in enumerate(frame.itertuples(index=False, name=None), 4):
        if i % 2 == 0:
            for col in range(1, ncol + 1):
                ws.cell(i, col).fill = PatternFill('solid', fgColor='F6F9FB')
        for j, val in enumerate(row, 1):
            v = safe_excel(val)
            c = ws.cell(i, j, v)
            if isinstance(v, (float, np.floating)):
                c.number_format = SIGNED4 if frame.columns[j-1] in signed_cols else NUM4
                c.font = Font(name='Aptos', size=10, color=GREEN)
            elif isinstance(v, (int, np.integer)) and not isinstance(v, bool):
                c.number_format = INT_FMT
                c.font = Font(name='Aptos', size=10, color=GREEN)
            else:
                c.font = Font(name='Aptos', size=10, color=DARK_BLUE)
            c.alignment = Alignment(vertical='center', horizontal='right' if isinstance(v, (float, int)) else 'left')
    ws.freeze_panes = 'C4' if ncol > 6 else 'B4'
    ws.sheet_view.showGridLines = False
    ws.auto_filter.ref = f'A3:{get_column_letter(ncol)}{max(3, len(frame)+3)}'
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.print_options.horizontalCentered = True
    return ws


def write_grouped_scores(sheet_name, b4_name, b3_name, title, b4_label, b3_label):

    ws = wb.create_sheet(sheet_name)
    ws.merge_cells('A1:G1')
    t = ws['A1']; t.value = title
    t.fill = PatternFill('solid', fgColor=DARK_BLUE)
    t.font = Font(name='Aptos', size=13, bold=True, color=WHITE)
    t.alignment = Alignment(vertical='center')
    ws.row_dimensions[1].height = 29
    ws.merge_cells('A3:A4')
    ws.merge_cells('B3:D3')
    ws.merge_cells('E3:G3')
    for address, label in [('A3', 'DR Severity'), ('B3', b4_label), ('E3', b3_label)]:
        c=ws[address]; c.value=label
        c.fill=PatternFill('solid', fgColor=DARK_BLUE)
        c.font=Font(name='Aptos', size=10, color=WHITE, bold=True)
        c.alignment=Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[3].height=37
    subheads=['With Vessel Mask Input', 'With Zero Vessel Mask Input', 'Difference']
    for j, head in enumerate(subheads, 2):
        for jj in (j, j+3):
            c=ws.cell(4,jj,head)
            c.fill=PatternFill('solid', fgColor=MID_BLUE)
            c.font=Font(name='Aptos', size=10, color=DARK_BLUE, bold=True)
            c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
    ws.row_dimensions[4].height=41
    names={'no_dr':'No_dr','mild':'Mild','moderate':'Moderate','severe':'Severe','proliferative_dr':'Proliferative'}
    for k, severity in enumerate(CLASS_NAMES):
        excel_row=k+5
        c=ws.cell(excel_row,1,names.get(severity,severity));c.font=Font(name='Aptos',size=11,bold=True,color=DARK_BLUE)
        if k%2 == 0:
            for j in range(1,8):ws.cell(excel_row,j).fill=PatternFill('solid',fgColor='F6F9FB')
        for model, first_col in ((b4_name,2), (b3_name,5)):
            filtered=per_score[(per_score.model==model)&(per_score.severity==severity)]
            if len(filtered)!=1:raise ValueError(f'Expected one score row for {model} / {severity}, got {len(filtered)}')
            row=filtered.iloc[0]
            for k2, field in enumerate(('avg_score_with_mask','avg_score_zero_mask','difference_with_minus_zero')):
                val=float(row[field]); c=ws.cell(excel_row,first_col+k2,val)
                c.number_format=SIGNED4 if k2==2 else NUM4
                c.font=Font(name='Aptos',size=11,color=GREEN)
                c.alignment=Alignment(horizontal='center',vertical='center')
        ws.row_dimensions[excel_row].height=24
    ws.column_dimensions['A'].width=22
    for x in 'BCDEFG':ws.column_dimensions[x].width=26
    ws.freeze_panes='B5'
    ws.sheet_view.showGridLines=False
    ws.print_options.horizontalCentered=True
    ws.sheet_properties.pageSetUpPr.fitToPage=True
    ws.sheet_view.zoomScale=85
    return ws

write_grouped_scores(
    'Scores_With_SE',
    'B0+B4+SE | refined','B0+B3+SE | predicted',
    'Average actual-class prediction scores - WITH SE attention',
    'With Refinement (B0+B4+SE)','Without Refinement (B0+B3+SE)')
write_grouped_scores(
    'Scores_Without_SE',
    'B0+B4 | refined | no SE','B0+B3 | predicted | no SE',
    'Average actual-class prediction scores - WITHOUT SE attention',
    'With Refinement (B0+B4)','Without Refinement (B0+B3)')

write_table('All_Severity_Averages', per_score,
            'All 4 models - average actual-class scores for each of the 5 DR severity classes',
            signed_cols=['difference_with_minus_zero'])
write_table('Overall_Metrics', overall,
            'All 4 models: actual mask vs zero mask - classification metrics')
write_table('Metric_Differences', comparison,
            'All 4 models: classification metric change (actual mask minus zero mask)',
            signed_cols=[x for x in comparison.columns if x.endswith('_difference')])
write_table('Per_Class_Metrics', per_metrics,
            'Precision, recall, F1-score and support - every DR severity class and mask condition')
write_table('Prediction_Switches', changes,
            'Class prediction changes when vessel masks are replaced with zero masks')
write_table('Confusion_Matrices', confusions,
            'Confusion matrix counts for actual-mask and zero-mask inference')

write_table('All_Image_Scores', pred_rows,
            'Every test-image result: real mask vs zero mask, true-class scores and all 5 class probabilities',
            signed_cols=['score_difference'])


for code,name in [
    ('no_dr','Images_No_DR'),('mild','Images_Mild'),('moderate','Images_Moderate'),
    ('severe','Images_Severe'),('proliferative_dr','Images_Proliferative')]:
    image_frame = pred_rows.loc[pred_rows['true_class'].eq(code)].copy()
    write_table(name, image_frame,
                'Individual test-image scores - '+code.replace('_',' ').title()+' - all four models',
                signed_cols=['score_difference'])

info=wb.create_sheet('Read_Me',0)
info.sheet_view.showGridLines=False
info.merge_cells('A1:D1')
info['A1']='Four-model zero vessel-mask evaluation: complete Excel results'
info['A1'].fill=PatternFill('solid',fgColor=DARK_BLUE)
info['A1'].font=Font(name='Aptos',size=14,bold=True,color=WHITE)
info.row_dimensions[1].height=30
notes=[
    ('Test images per model', str(len(PREDICTIONS['B0+B3+SE | predicted']['labels']))),
    ('Models', 'B0+B3+SE predicted; B0+B3 predicted without SE; B0+B4+SE refined; B0+B4 refined without SE'),
    ('With-mask condition','Saved trained checkpoint with its original predicted or GAN-refined vessel mask'),
    ('Zero-mask condition','Same checkpoint and same RGB image, but the vessel-mask input is all zeros at testing'),
    ('Average per severity','Mean probability assigned to the actual DR class across images belonging to that class'),
    ('Score Difference','With-mask average minus zero-mask average; positive means higher average correct-class score with mask'),
    ('Accuracy Difference','With-mask test accuracy minus zero-mask test accuracy, for the same trained checkpoint'),
    ('Not measured','Newly trained zero-mask models; vessel-only classifiers; causal effect of masks on Stage 1 versus Stage 3'),
    ('Precision','Values keep raw numerical precision; Excel displays four decimal places'),
    ('Where to find means','Scores_With_SE; Scores_Without_SE; All_Severity_Averages'),
    ('Where to find image-level values','All_Image_Scores and Images_No_DR / Images_Mild / Images_Moderate / Images_Severe / Images_Proliferative'),
    ('Where to find classification metrics','Overall_Metrics; Metric_Differences; Per_Class_Metrics'),
    ('Google Drive folder',str(RESULT_DIR)),
]
for r,(label,value) in enumerate(notes,3):
    info.cell(r,1,label).font=Font(name='Aptos',size=10,bold=True,color=DARK_BLUE)
    info.merge_cells(start_row=r,start_column=2,end_row=r,end_column=4)
    info.cell(r,2,value).font=Font(name='Aptos',size=10,color=GRAY)
    info.cell(r,2).alignment=Alignment(wrap_text=True,vertical='center')
    info.row_dimensions[r].height=31 if len(value)>65 else 23
info.column_dimensions['A'].width=34
info.column_dimensions['B'].width=45
info.column_dimensions['C'].width=29
info.column_dimensions['D'].width=30
info.freeze_panes='B3'

wb.save(EXCEL_FILE)

check=load_workbook(EXCEL_FILE,read_only=True,data_only=True)
assert check['Scores_With_SE'].max_row==9
assert check['Scores_Without_SE'].max_row==9
assert check['All_Severity_Averages'].max_row==23
assert check['All_Image_Scores'].max_row==len(pred_rows)+3
assert len(set(per_score['severity']))==5
assert len(per_score)==20
assert len(overall)==8
check.close()
print('\n'+'='*95)
print('EXCEL WORKBOOK SAVED TO GOOGLE DRIVE:')
print(EXCEL_FILE)
print('Contains: grouped 5-class score averages (with and without SE), 4-model average scores,')
print('overall and per-class metrics, and every test-image score in all five DR severities.')
print('The Excel cells preserve full numerical values and display scores to 4 decimal places.')
print('='*95)



EXCEL WORKBOOK SAVED TO GOOGLE DRIVE:
/content/drive/MyDrive/DR_Project/zero_mask_all_four_results/Zero_Mask_Four_Models_Full_Results.xlsx
Contains: grouped 5-class score averages (with and without SE), 4-model average scores,
overall and per-class metrics, and every test-image score in all five DR severities.
The Excel cells preserve full numerical values and display scores to 4 decimal places.
